---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [14]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: d:\Facultate\ADC\AI_Engineering\echochamber-project-team-1
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("D:\Facultate\ADC\AI_Engineering\echochamber-project-team-1\data\cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[NicusorDanRO] Felicitări , BRAVO tuturor celor care au avut curajul de a ieși public, de a spu
[StirileProTV] In nicio țară polițiștele nu merg la.servici cu părul.desetit ca pe.bulevard.Dac
[expertforum] BRAVO ! Baga le Rachete de mezzo raggio vecine la granita, sì apoi plangete ca N


<>:4: SyntaxWarning: invalid escape sequence '\F'
<>:4: SyntaxWarning: invalid escape sequence '\F'
C:\Users\mihai\AppData\Local\Temp\ipykernel_12052\3725691285.py:4: SyntaxWarning: invalid escape sequence '\F'
  corpus = pd.read_json("D:\Facultate\ADC\AI_Engineering\echochamber-project-team-1\data\cleaned/corpus_youtube_sample.jsonl", lines=True)


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [10]:
# modifica dupa preferinte

AXA_1 = "epistemic"
AXA_2 = "geopolitic"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [11]:
AXA_1_DEFINITION = """
epistemic_frame măsoară dacă textul contestă cunoașterea oficială,
expertiza instituțională, datele științifice sau autoritatea epistemică
a elitelor (experți, academici, instituții, statistici oficiale).

0 = absent: textul nu discută adevărul, expertiza, sursele sau cunoașterea oficială
1 = prezent: textul exprimă scepticism, neîncredere sau contestare punctuală
2 = dominant: textul este organizat în jurul ideii că adevărul este ascuns,
manipulat sau controlat de elite, experți, presă, instituții sau sistem
"""
AXA_2_DEFINITION = """
geopolitical_frame măsoară dacă textul interpretează politica prin
raportare la actori externi, conflicte geopolitice, suveranitate,
interferență străină sau opoziții între blocuri de putere
(ex. UE, NATO, Rusia, SUA, globalism).

0 = absent: textul nu folosește o interpretare geopolitică
1 = prezent: textul menționează actori externi sau influențe externe
2 = dominant: textul explică problema politică principală prin control extern,
dependență, trădare națională, globalism, UE, NATO, SUA, Rusia sau alte blocuri
de putere
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [12]:
MINI_PROMPT = f"""
Ești un model de adnotare pentru analiză de discurs politic.

SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}

CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2

DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Nu explica decizia.
8. Returnează doar JSON valid.

FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un model de adnotare pentru analiză de discurs politic.

SARCINĂ:
Adnotează comentariul folosind două axe:
1. epistemic
2. geopolitic

CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
epistemic = 0 / 1 / 2
geopolitic = 0 / 1 / 2

DEFINIȚII:

epistemic_frame măsoară dacă textul contestă cunoașterea oficială,
expertiza instituțională, datele științifice sau autoritatea epistemică
a elitelor (experți, academici, instituții, statistici oficiale).

0 = absent: textul nu discută adevărul, expertiza, sursele sau cunoașterea oficială
1 = prezent: textul exprimă scepticism, neîncredere sau contestare punctuală
2 = dominant: textul este organizat în jurul ideii că adevărul este ascuns,
manipulat sau controlat de elite, experți, presă, instituții sau sistem


geopolitical_frame măsoară dacă textul interpretează poli

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [18]:
TESTS = corpus.sample(10)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
317,yt_jxIZUCUPnAw_Ugwzh1OwH0iAS_vITHh4AaABAg,turcescu111,"A fost rupere în audiență, pe asta n-au cum s-...","Pai daca el a promis ucrainenilor, cum zici tu..."
240,yt_yEuctxNb4O0_Ugw0iVv7HakoRysTqWp4AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,Politicienii mină in mină cu justiția să-i sca...
363,yt_DKhN-ua4lyw_Ugy-Tn8ntNbT8dD0pM94AaABAg,georgesimionoficial,#gs #georgesimion #democratie #impreuna #prosp...,Haideți să ne luam țara înapoi CĂLIN GEORGESCU...
195,yt_I09VxUz3YxQ_UgwMEbeoGeiIifXY2Il4AaABAg,AltcevacuAdrianArtene,MARILE ADEVĂRURI despre SADOVEANU. Moștenitoru...,Abordarea asupra Masoneriei este foarte superf...
375,yt_yEuctxNb4O0_Ugx1z3x0Sui0lSBOW3h4AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,Felicitari dragilor! Datorita curajului vostru...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [15]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [19]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [20]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Pai daca el a promis ucrainenilor, cum zici tu Robert, ?el executa ce i se ordona de cei care i.au pus

OUTPUT MODEL:
```json
{
  "target": "politica externă a României față de Ucraina",
  "stance": "anti",
  "tone": "acuzator",
  "epistemic": 1,
  "geopolitic": 2
}
```
COMENTARIU:
Politicienii mină in mină cu justiția să-i scape de pedepse,ei sunt primii care fură si au nevoie de protecția justiției, ex cazul Nordis,cazul Vanghelie,și multe altele

OUTPUT MODEL:
```json
{
  "target": "justiția",
  "stance": "anti",
  "tone": "acuzator",
  "epistemic": 2,
  "geopolitic": 0
}
```
COMENTARIU:
Haideți să ne luam țara înapoi CĂLIN GEORGESCU PREȘEDINTE POPORULUI ROMÂN

OUTPUT MODEL:
```json
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "mobilizator",
  "epistemic": 0,
  "geopolitic": 0
}
```
COMENTARIU:
Abordarea asupra Masoneriei este foarte superficială. Faptul că jurământul se făcea cu mâna pe Biblie, nu are nicio relevanţă, câtă vreme şi vrăjitoarele fac vrăj

In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

Am ales axele "epistemic" și "geopolitic" pentru a surprinde contestarea expertizei și interpretările bazate pe influențe externe;
Modelul a returnat aproape mereu JSON corect, dar uneori în blocuri Markdown;
Principala problemă a fost suprainterpretarea target-ului și a axei epistemice;
Aș adăuga alte reguli pentru identificarea target-ului și pentru folosirea valorii 2 la epistemic.